#### Feature Engineering

##### Importing Libraries and Loading Data

In [42]:
import numpy as np
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
processed_dir = project_root /'data'/'processed'

long = pd.read_parquet(processed_dir/'station_time.parquet')
station_dim = pd.read_parquet(processed_dir /'station_dimension.parquet')

In [10]:
qcols = [c for c in long['quarter_hour'].unique()]
qcols = sorted(qcols, key=lambda c: (c[:2] < '05', c))

In [11]:
# Making a Pivot Table Frame in Chronological Time Order

wide = long.pivot_table(
    index=['nlc', 'daytype', 'metric'],
    columns='quarter_hour',
    values='count',
)

In [12]:
wide = wide[qcols]

In [14]:
wide.head()

quarter_hour         0500-0515  0515-0530  0530-0545  0545-0600  0600-0615  \
nlc daytype metric                                                           
500 fri     entries  11.125288  18.511613  28.829019  41.701635  54.004743   
            exits     2.320267   8.407356  12.947226  22.672407  39.956265   
    mon     entries   7.657625  14.688244  26.307872  40.167666  55.685555   
            exits     0.525849   6.062292   9.310302  22.566779  40.963425   
    sat     entries   7.826679  11.903178  18.346930  25.201519  32.379213   

quarter_hour         0615-0630  0630-0645   0645-0700   0700-0715   0715-0730  \
nlc daytype metric                                                              
500 fri     entries  64.747046  84.092904  106.078971  128.011669  155.298812   
            exits    59.743767  77.170189   97.200499  116.894328  132.499678   
    mon     entries  72.566382  95.719089  121.125659  146.429966  182.070004   
            exits    62.026847  80.504040  103.720505  126.206181  144.940724   
    sat     entries  36.147691  42.777563   49.048225   51.751982   53.743946   

quarter_hour         ...  0230-0245  0245-0300  0300-0315  0315-0330  \
nlc daytype metric   ...                                               
500 fri     entries  ...   2.540328   2.659495   2.995062   3.551773   
            exits    ...  17.233300  14.014180  14.258489  11.349448   
    mon     entries  ...   0.000000   0.000000   0.000000   0.000000   
            exits    ...   0.000000   0.000000   0.000000   0.000000   
    sat     entries  ...   3.327380   3.036721   2.879388   3.665862   

quarter_hour         0330-0345  0345-0400  0400-0415     0415-0430  \
nlc daytype metric                                                   
500 fri     entries   3.735997   4.915143   4.882796  9.717703e+00   
            exits    11.963900  10.205119  10.379331  8.346284e+00   
    mon     entries   0.000000   0.000000   0.000000  4.817381e+00   
            exits     0.000000   0.000000   0.000000  1.381897e-07   
    sat     entries   3.237395   4.111583   4.678346  7.420842e+00   

quarter_hour            0430-0445     0445-0500  
nlc daytype metric                               
500 fri     entries  8.503073e+00  7.931928e+00  
            exits    7.545557e+00  4.355307e+00  
    mon     entries  4.817381e+00  4.817381e+00  
            exits    1.381897e-07  1.381897e-07  
    sat     entries  6.350199e+00  6.006688e+00  

[5 rows x 96 columns]

In [15]:
wide.shape

(4710, 96)

In [16]:
# Creating Time-Window Column Groups

def in_window(prefix):
    return [c for c in qcols if c[:2] in prefix]

In [17]:
early = in_window(('05', '06'))
am = in_window(('07', '08', '09'))
mid = in_window(('10', '11', '12', '13', '14', '15'))
pm = in_window(('16', '17', '18'))
eve = in_window(('19', '20', '21'))
late = in_window(('22', '23', '00', '01', '02', '03', '04'))

In [18]:
active_nlc = station_dim[station_dim['active']]['nlc']

In [43]:
# Creating a Features matrix and Calculating features per station

records = {}
for nlc in active_nlc:
    twt_entries = wide.loc[(nlc, 'twt', 'entries')]
    twt_exits = wide.loc[(nlc, 'twt', 'exits')]
    sat_entries = wide.loc[(nlc, 'sat', 'entries')]

    twt_total = twt_entries.sum()
    sat_total = sat_entries.sum()

    twt_profile = twt_entries / twt_total
    sat_profile = sat_entries / sat_total
    f = {}

    # Log transforming the total because there is a big size difference between stations
    f['log_total_weekday'] = np.log1p(twt_total)

    f['early_share'] = twt_profile[early].sum()
    f['eve_share'] = twt_profile[eve].sum()
    f['late_share'] = twt_profile[late].sum()

    f['peakness'] = twt_profile.max()

    am_e, am_x = twt_entries[am].sum(), twt_exits[am].sum()
    pm_e, pm_x = twt_entries[pm].sum(), twt_exits[pm].sum()

    f['am_asym'] = (am_e - am_x) / (am_e + am_x)
    f['pm_asym'] = (pm_e - pm_x) / (pm_e + pm_x)

    f['weekend_shift'] = np.abs(twt_profile - sat_profile).sum() / 2
    f['weekend_ratio'] = sat_total / twt_total

    records[nlc] = f

features = pd.DataFrame(records).T
features.index.name = 'nlc'

In [45]:
features.head()

,log_total_weekday,early_share,eve_share,late_share,peakness,am_asym,pm_asym,weekend_shift,weekend_ratio
nlc,,,,,,,,,
750,7.760586,0.077621,0.094472,0.041111,0.030217,0.139383,-0.006690,0.213420,0.783216
1404,7.923830,0.071789,0.080898,0.027622,0.036185,0.116857,-0.139332,0.267944,0.725684
3000,8.666488,0.072965,0.052739,0.012879,0.043083,0.420971,-0.182355,0.291676,0.847376
500,9.014619,0.054701,0.059517,0.020899,0.040084,0.274624,-0.098968,0.242156,0.812725
502,9.483445,0.018862,0.125731,0.043576,0.049968,-0.556020,0.334524,0.255896,0.527519


In [46]:
features.shape

(432, 9)

In [47]:
# Saving DataFrame
features.to_parquet(processed_dir/'station_features.parquet')

In [48]:
features

,log_total_weekday,early_share,eve_share,late_share,peakness,am_asym,pm_asym,weekend_shift,weekend_ratio
nlc,,,,,,,,,
750,7.760586,0.077621,0.094472,0.041111,0.030217,0.139383,-0.006690,0.213420,0.783216
1404,7.923830,0.071789,0.080898,0.027622,0.036185,0.116857,-0.139332,0.267944,0.725684
3000,8.666488,0.072965,0.052739,0.012879,0.043083,0.420971,-0.182355,0.291676,0.847376
500,9.014619,0.054701,0.059517,0.020899,0.040084,0.274624,-0.098968,0.242156,0.812725
502,9.483445,0.018862,0.125731,0.043576,0.049968,-0.556020,0.334524,0.255896,0.527519
...,...,...,...,...,...,...,...,...,...
6560,10.403914,0.018856,0.134584,0.041981,0.044482,-0.344185,0.310611,0.256120,0.620645
5131,10.098654,0.170494,0.035875,0.007407,0.055914,0.732043,-0.586005,0.367480,0.673885
6561,9.709958,0.076326,0.066574,0.014392,0.028572,0.128118,0.100825,0.202767,0.864295


In [ ]:
mon_